# Revenue Forecasting
 

# 1. Import libraries

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# 2. import the cleaned daily_model table

In [11]:
df = pd.read_csv("../data_clean/daily_model_table_clean.csv",parse_dates=["date"])

In [13]:
df.sample(5)

,date,revenue,orders,average_order_value,orders_lunch,orders_afternoon,orders_dinner,orders_dine_in,orders_delivery,orders_takeaway,...,restaurant_day_type,holiday_name,is_holiday,temperature_avg_c,temp_max_c,temp_min_c,precipitation_mm,is_rainy,weather_condition,is_promo_day
278,2024-10-05,4612.90,119,38.76,30,10,79,74,32,13,...,Peak,NaN,False,14.5,16.6,12.8,0.0,False,Normal,False
620,2025-09-12,4031.20,95,42.43,21,6,68,69,17,9,...,Peak,NaN,False,16.4,21.3,12.3,0.0,False,Normal,False
414,2025-02-18,1609.20,45,35.76,12,7,26,30,8,7,...,Quiet,NaN,False,-0.2,2.0,-1.9,0.0,False,Cold,False
144,2024-05-24,3186.60,83,38.39,20,8,55,49,26,8,...,Peak,NaN,False,11.9,13.2,10.7,3.8,True,Rain,False
539,2025-06-23,2094.68,61,34.34,21,4,36,36,12,13,...,Quiet,NaN,False,14.4,20.1,9.8,0.0,False,Normal,True


# 3. Train/Test Split
   - chronological split
   - NO random train_test_split

In [14]:
# ============================================
# TRAIN / TEST SPLIT
# ============================================

df = df.copy()

df = df.sort_values("date").reset_index(drop=True)

# Use the last 90 days as the test period
test_days = 90

train_df = df.iloc[:-test_days].copy()
test_df = df.iloc[-test_days:].copy()

print("Training period:")
print(train_df["date"].min(), "to", train_df["date"].max())

print("\nTest period:")
print(test_df["date"].min(), "to", test_df["date"].max())

print("\nTraining observations:", len(train_df))
print("Test observations:", len(test_df))

Training period:
2024-01-01 00:00:00 to 2025-10-02 00:00:00

Test period:
2025-10-03 00:00:00 to 2025-12-31 00:00:00

Training observations: 641
Test observations: 90


## 4. Evaluation Metrics
   - MAE
   - RMSE
   - MAPE

In [16]:
# ============================================
# 6. EVALUATION METRICS
# ============================================

from sklearn.metrics import mean_absolute_error, mean_squared_error

def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    mape = np.mean(
        np.abs((y_true - y_pred) / y_true)
    ) * 100

    return {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE (%)": mape
    }

## 5. Baseline Models
   - Previous Day
   - Seasonal Naive
   - Weekday Mean

In [17]:
# ============================================
# 7. BASELINE MODELS
# ============================================

# --------------------------------------------
# Baseline 1: Previous Day
# --------------------------------------------

naive_predictions = np.repeat(
    train_df["revenue"].iloc[-1],
    len(test_df)
)


# --------------------------------------------
# Baseline 2: Same Weekday Last Week
# --------------------------------------------

seasonal_predictions = []

history = train_df.copy()

for date in test_df["date"]:
    previous_week = date - pd.Timedelta(days=7)

    value = history.loc[
        history["date"] == previous_week,
        "revenue"
    ]

    if len(value) > 0:
        seasonal_predictions.append(value.iloc[0])
    else:
        seasonal_predictions.append(
            history["revenue"].iloc[-1]
        )

    # Add actual observation after prediction
    # so the next prediction has access to
    # newly observed data
    history = pd.concat(
        [history, test_df[test_df["date"] == date]]
    )


# --------------------------------------------
# Baseline 3: Historical Weekday Mean
# --------------------------------------------

weekday_means = (
    train_df
    .groupby("day_of_week_num")["revenue"]
    .mean()
)

weekday_predictions = (
    test_df["day_of_week_num"]
    .map(weekday_means)
    .to_numpy()
)

In [18]:
baseline_results = []

baseline_results.append({
    "Model": "Naive - Previous Day",
    **calculate_metrics(
        test_df["revenue"].to_numpy(),
        naive_predictions
    )
})

baseline_results.append({
    "Model": "Seasonal Naive - Previous Week",
    **calculate_metrics(
        test_df["revenue"].to_numpy(),
        seasonal_predictions
    )
})

baseline_results.append({
    "Model": "Weekday Mean",
    **calculate_metrics(
        test_df["revenue"].to_numpy(),
        weekday_predictions
    )
})

baseline_results_df = pd.DataFrame(baseline_results)

baseline_results_df.sort_values("MAE")

,Model,MAE,RMSE,MAPE (%)
2,Weekday Mean,375.624744,486.581958,14.478338
1,Seasonal Naive - Previous Week,514.204000,691.989711,19.597877
0,Naive - Previous Day,913.811111,1165.487736,33.918617


## 6. Prophet
   - Revenue only
   - + Calendar
   - + Weather
   - + Promotions

## 9. SARIMA / SARIMAX

## 10. Random Forest

## 11. Model Comparison

## 12. Best Model

## 13. Forecast Future Revenue

## 14. Business Interpretation